# 08 — Query Plans & Indexes

## 1. Why this concept exists

A database query is a **request**, not a step-by-step program. The engine chooses an execution strategy: which table to read first, whether to use an index, and how to access rows.

An index is an additional data structure that can make selective lookups cheaper, but it also has storage and write-maintenance cost.

## 2. Short mental model

Think of a table as a book:

- **Full table scan** → read every page looking for matching rows.
- **B-tree index** → use an ordered lookup structure to find relevant row locations.
- **Query plan** → the engine's chosen route from SQL to result.

The important engineering question is not *"Did I create an index?"* but:

> **Did the index change the work the database has to do for the workload that matters?**

We will use SQLite from Python's standard library, so this notebook runs in a fresh Colab runtime without installing packages.

## Capability contract

**Capability:** C02 — Data Access Querying & Modeling  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Translate a business question into correct data access and defend the model/query behavior.


## 🖊️ Sketch note

![Sketch note — Query Plans & Indexes](../assets/sketch-notes/08-query-plans-indexes.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S07 — Query plans and indexing (DATA-1102)** → [`../work_simulations/07_acme_commerce_latency_budget/README.md`](../work_simulations/07_acme_commerce_latency_budget/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Learning objectives

- read the main operators in a simple plan;
- connect a filter to an access path;
- compare before/after measurements;
- explain an index trade-off.


## Predict before you run

Predict which operation will dominate work and what an index should change.


In [ ]:
import sqlite3
import time
from pathlib import Path
import tempfile

db_path = Path(tempfile.gettempdir()) / "query_plans_indexes.db"
if db_path.exists():
    db_path.unlink()

con = sqlite3.connect(db_path)
con.execute("PRAGMA journal_mode=WAL")
con.execute("""
CREATE TABLE events (
    event_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    event_type TEXT NOT NULL,
    event_date TEXT NOT NULL,
    amount REAL NOT NULL
)
""")

rows = [
    (i, (i % 1000) + 1, ["purchase", "refund", "login"][i % 3],
     f"2026-01-{(i % 28) + 1:02d}", float((i * 17) % 500 + 10))
    for i in range(1, 100_001)
]
con.executemany(
    "INSERT INTO events(event_id, customer_id, event_type, event_date, amount) VALUES (?, ?, ?, ?, ?)",
    rows
)
con.commit()
print("rows:", con.execute("SELECT COUNT(*) FROM events").fetchone()[0])

## 3. Inspect the plan before optimizing

SQLite exposes its chosen plan through `EXPLAIN QUERY PLAN`.

Start with a selective lookup by `customer_id`. Before an index exists, the natural baseline is a scan.

In [ ]:
def show_plan(sql, params=()):
    print("SQL:", sql)
    for row in con.execute("EXPLAIN QUERY PLAN " + sql, params):
        print(row)

query = "SELECT event_id, event_date, amount FROM events WHERE customer_id = ?"
show_plan(query, (777,))

In [ ]:
def timed_query(sql, params=(), repeats=20):
    con.execute(sql, params).fetchall()
    start = time.perf_counter()
    for _ in range(repeats):
        con.execute(sql, params).fetchall()
    return (time.perf_counter() - start) / repeats

before = timed_query(query, (777,))
print(f"Average before index: {before * 1000:.3f} ms")

## 4. Implement the index

A B-tree index keeps keys ordered so the engine can navigate to matching values instead of inspecting every row.

Create an index that matches the lookup predicate.

In [ ]:
con.execute("CREATE INDEX idx_events_customer_id ON events(customer_id)")
con.commit()

show_plan(query, (777,))
after = timed_query(query, (777,))
print(f"Average after index:  {after * 1000:.3f} ms")
print(f"Observed speed ratio (before / after): {before / after:.2f}x")

## 5. Measure more than one query

An index is useful only relative to a workload.

Compare a selective lookup with a query that returns a large fraction of the table. Also notice that a query can use an index without becoming magically cheap.

In [ ]:
queries = {
    "selective": ("SELECT * FROM events WHERE customer_id = ?", (777,)),
    "broad": ("SELECT * FROM events WHERE event_type = ?", ("purchase",)),
}

for name, (sql, params) in queries.items():
    print(f"\n{name}")
    show_plan(sql, params)
    ms = timed_query(sql, params) * 1000
    print(f"average: {ms:.3f} ms")

## 6. Break it deliberately

Drop the index and inspect the plan again.

Then create a composite index `(customer_id, event_date)` and test a query filtering by both columns.

The key lesson: **index column order matters**. An index is not a generic accelerator; its usefulness depends on the predicates and ordering in the workload.

In [ ]:
con.execute("DROP INDEX idx_events_customer_id")
con.commit()
print("After dropping the index:")
show_plan(query, (777,))

con.execute("CREATE INDEX idx_events_customer_date ON events(customer_id, event_date)")
con.commit()

composite_query = """
SELECT event_id, event_date, amount
FROM events
WHERE customer_id = ? AND event_date >= ?
ORDER BY event_date
"""
print("\nComposite-index query:")
show_plan(composite_query, (777, "2026-01-10"))

## 7. Diagnose what happened

Ask these questions when an index experiment surprises you:

1. What rows does the query actually need?
2. How selective is the predicate?
3. Does the plan show a scan or an index search?
4. Does the index match the predicate/order?
5. What is the write and storage cost of maintaining it?

**Engineering challenge:** create a composite index for `(event_type, event_date)`, compare the plan for a date-filtered purchase query, and explain why the result changed (or did not).

## 8. Production implications

Indexes are a workload decision.

In production systems, consider:

- read latency and throughput;
- write amplification and ingestion cost;
- index storage;
- cardinality/selectivity;
- query patterns and plan stability;
- stale statistics and optimizer behavior;
- operational cost of adding/removing indexes.

The production habit to build here is simple:

> **Measure the plan and workload before declaring an optimization successful.**

In [ ]:
baseline = con.execute(
    "SELECT event_id, event_date, amount FROM events WHERE customer_id = ? ORDER BY event_id",
    (777,)
).fetchall()
optimized = con.execute(
    "SELECT event_id, event_date, amount FROM events INDEXED BY idx_events_customer_date WHERE customer_id = ? ORDER BY event_id",
    (777,)
).fetchall()
assert baseline == optimized
print("Invariant passed: index changed access strategy without changing results.")